# Build GroundTruth


In [20]:
import pandas as pd
data = pd.read_excel("../data/df_processed.xlsx")
df_gt = data.copy()
# Tạo cột văn bản gộp  và chuyển về chữ thường
df_gt['full_text_search'] = (df_gt['title'].astype(str) + " " + 
                             df_gt['description'].astype(str)  + " " + 
                             df_gt['requirements'].astype(str) + " " +
                             df_gt['benefit'].astype(str)).str.lower()
df_gt['full_text_search'].head()


0    tuyển kế toán tổng hợp đi làm ngay thu nhập up...
1    tuyển business development executive làm việc ...
2    tuyển nhân viên content community marketing là...
3    tuyển trưởng nhóm kinh doanh làm việc tại công...
4    tuyển nhân viên kinh doanh giải pháp cntt làm ...
Name: full_text_search, dtype: object

In [21]:
complex_test_set = pd.read_json("queries_natural_200.json")
print(complex_test_set.head())

                                               query  \
0  Tìm vị trí staff kỹ sư cầu đường làm việc tại ...   
1  Tìm vị trí chuyên viên kế toán tổng hợp làm vi...   
2             Tìm việc làm xây dựng khu vực Đồng Nai   
3  Tìm vị trí chuyên viên chuyên môn ngân hàng là...   
4            Job môi giới bất động sản thủ đô Hà Nội   

                                       must_have  min_match  
0  [Nhân viên, Kỹ sư cầu đường, TP. Hồ Chí Minh]          2  
1  [Nhân viên, Kế toán tổng hợp, TP Hồ Chí Minh]          2  
2                           [Xây dựng, Đồng Nai]          1  
3      [Nhân viên, Chuyên môn Ngân hàng, Hà Nội]          2  
4                [Môi giới bất động sản, Hà Nội]          1  


In [22]:
from tqdm import tqdm
ground_truth_map = {} 
ground_truth_stats = [] # Để lưu thống kê xem query nào tìm được bao nhiêu job

print(f"Bắt đầu tạo Ground Truth cho {len(complex_test_set)} queries...")

# for q_idx, test_case in enumerate(tqdm(complex_test_set)):
#     keywords = [k.lower() for k in test_case['must_have']]
#     min_match = test_case['min_match']

for q_idx, row in tqdm(complex_test_set.iterrows(), total=len(complex_test_set)):
    query_text = row['query']
    # Xử lý list must_have: đưa tất cả về chữ thường
    keywords = [k.lower() for k in row['must_have']]
    min_match = row['min_match']

    
    # --- ĐẾM TỪ KHÓA ---
    def is_relevant(text):
        if not isinstance(text, str): return False
        text_lower = text.lower() # Chuyển văn bản gốc về chữ thường để khớp
        # Đếm số từ khóa xuất hiện trong văn bản
        match_count = sum(1 for kw in keywords if kw in text_lower)
        return match_count >= min_match

    # Lọc ra các dòng thỏa mãn điều kiện
    relevant_jobs = df_gt[df_gt['full_text_search'].apply(is_relevant)]
    
    # Lấy tập hợp các ID
    relevant_ids = set(relevant_jobs['id'].tolist())
    
    # Lưu vào map
    ground_truth_map[q_idx] = relevant_ids
    
    # Lưu thống kê
    ground_truth_stats.append({
        "query_id": q_idx,
        "query_text": query_text,
        "relevant_count": len(relevant_ids),
        "keywords": str(keywords)
    })

print("\n✅ HOÀN TẤT XÂY DỰNG GROUND TRUTH!")


Bắt đầu tạo Ground Truth cho 205 queries...


100%|██████████| 205/205 [00:16<00:00, 12.34it/s]


✅ HOÀN TẤT XÂY DỰNG GROUND TRUTH!


In [23]:
# Chuyển thống kê sang DataFrame để dễ nhìn
df_stats = pd.DataFrame(ground_truth_stats)

# Hiển thị các query tìm được ít kết quả quá (Cần cảnh báo)
low_result_queries = df_stats[df_stats['relevant_count'] < 5]
if not low_result_queries.empty:
    print(f"⚠️ CẢNH BÁO: Có {len(low_result_queries)} query có quá ít đáp án đúng (< 5 jobs).")
    print("Bạn nên nới lỏng từ khóa hoặc giảm 'min_match' cho các query này:")
    display(low_result_queries[['query_id', 'query_text', 'relevant_count', 'keywords']])
else:
    print("👍 Tất cả query đều có đủ dữ liệu để đánh giá.")

print("\nTop 5 query có nhiều đáp án đúng nhất:")
display(df_stats.sort_values('relevant_count', ascending=False).head(5))

👍 Tất cả query đều có đủ dữ liệu để đánh giá.

Top 5 query có nhiều đáp án đúng nhất:


,query_id,query_text,relevant_count,keywords
54,54,Cần tìm job đào tạo ở Hà Nội,3721,"['đào tạo', 'hà nội']"
7,7,Tìm vị trí nhân viên telesales,3316,"['nhân viên', 'telesales']"
107,107,Cần tìm job xây dựng ở hồ chí minh,2583,"['xây dựng', 'hồ chí minh']"
2,2,Tìm việc làm xây dựng khu vực Đồng Nai,2464,"['xây dựng', 'đồng nai']"
68,68,Tuyển nhân viên môi trường ở tp. hcm,1866,"['nhân viên', 'môi trường', 'tp. hcm']"


In [24]:
print(ground_truth_map)

{0: {4031, 3458, 5185, 4932, 2565, 870, 40, 5465, 3577, 5362, 115, 3348, 439, 601, 6010, 91, 95}, 1: {0, 1025, 1028, 4614, 1044, 1557, 1046, 1558, 1049, 1562, 1565, 5149, 1058, 1059, 1573, 1064, 5163, 562, 1595, 5179, 1605, 5189, 1611, 1104, 1106, 1619, 85, 1624, 1625, 90, 1115, 91, 5732, 1125, 1130, 1643, 1132, 5741, 114, 1138, 1139, 3190, 5289, 1198, 1209, 1216, 1217, 1218, 1219, 1742, 1267, 1276, 1279, 1280, 1293, 1295, 1834, 821, 1334, 1344, 1351, 5448, 1355, 5970, 1366, 1374, 1896, 6010, 1420, 1422, 1433, 1435, 5021, 1439, 6047, 5540, 1448, 1449, 1450, 1456, 5053, 6092, 6097, 1492, 1498, 1510}, 2: {2, 3, 5, 6, 8, 9, 11, 15, 18, 20, 22, 23, 24, 25, 26, 28, 30, 33, 34, 35, 36, 37, 40, 42, 46, 47, 49, 54, 56, 59, 61, 67, 69, 70, 71, 73, 75, 76, 78, 79, 80, 81, 91, 93, 95, 98, 100, 105, 107, 108, 111, 115, 117, 125, 126, 127, 128, 131, 134, 138, 140, 151, 152, 154, 155, 156, 157, 159, 160, 161, 163, 167, 169, 173, 174, 175, 178, 181, 182, 186, 187, 189, 195, 196, 202, 207, 210, 216, 2

In [25]:
import pickle

with open('groundtruth200.pkl', 'wb') as f:
    pickle.dump(ground_truth_map, f)
print("Đã lưu file 'groundtruth200.pkl'")

Đã lưu file 'groundtruth200.pkl'


# Evaluate with GridSearch



In [26]:
with open('groundtruth200.pkl', 'rb') as f:
    loaded_groundtruth = pickle.load(f)

In [41]:
import pandas as pd
import numpy as np

def calculate_metrics_optimized(group, ground_truth):
    # Lấy thông tin index
    if isinstance(group.name, tuple):
        model_name, q_id = group.name
    else:
        q_id = group.name
    
    # 1. Sắp xếp kết quả trả về theo thứ hạng (rank)
    group = group.sort_values('rank')
    relevants = group['label'].values 
    
    # 2. Lấy số lượng Ground Truth thực tế
    true_set = ground_truth.get(q_id, set())
    total_true_items = len(true_set)
    
    metrics = {}
    
    # Nếu query này không có ground truth nào, coi như các chỉ số = 0 để tránh chia cho 0
    if total_true_items == 0:
        return pd.Series({ 'MAP': 0, 'MRR': 0, 'Hit@5': 0, 'P@5': 0, 'NDCG@5': 0, 'MRR@5': 0, 'MAP@5': 0,
                           'Hit@10': 0, 'P@10': 0, 'NDCG@10': 0, 'MRR@10': 0, 'MAP@10': 0,
                           'Hit@20': 0, 'P@20': 0, 'NDCG@20': 0, 'MRR@20': 0, 'MAP@20': 0 })

    # --- Pre-calculation (Tính trước các mảng) ---
    ranks = np.arange(len(relevants)) + 1
    cumulative_relevant = np.cumsum(relevants)
    precisions_at_i = cumulative_relevant / ranks
    discounts = np.log2(ranks + 1)
    
    # ====================================================
    # A. GLOBAL METRICS (Dựa trên toàn bộ list trả về)
    # ====================================================
    metrics['MAP'] = np.sum(precisions_at_i * relevants) / total_true_items

    try:
        first_rel_idx = np.argmax(relevants == 1)
        if relevants[first_rel_idx] == 1:
            metrics['MRR'] = 1 / (first_rel_idx + 1)
        else:
            metrics['MRR'] = 0
    except:
        metrics['MRR'] = 0

    # ====================================================
    # B. METRICS TẠI CÁC NGƯỠNG K
    # ====================================================
    cutoffs = [5, 10, 20]
    
    for k in cutoffs:
        actual_k = min(k, len(relevants))
        r_k = relevants[:actual_k]
        
        # 1. Hit Ratio@K
        metrics[f'Hit@{k}'] = 1 if np.any(r_k) else 0
        
        # 2. Precision@K
        metrics[f'P@{k}'] = cumulative_relevant[actual_k - 1] / k if actual_k > 0 else 0
        
        # 3. NDCG@K (Logic chuẩn mực)
        dcg = np.sum(r_k / discounts[:actual_k])
        ideal_r_k = np.zeros(actual_k)
        num_ones_possible = min(actual_k, total_true_items)
        ideal_r_k[:num_ones_possible] = 1 
        idcg = np.sum(ideal_r_k / discounts[:actual_k])
        metrics[f'NDCG@{k}'] = dcg / idcg if idcg > 0 else 0
            
        # 4. MRR@K
        first_idx_k = np.argmax(r_k == 1)
        if r_k[first_idx_k] == 1:
            metrics[f'MRR@{k}'] = 1 / (first_idx_k + 1)
        else:
            metrics[f'MRR@{k}'] = 0
            
        # 5. MAP@K (ĐÃ SỬA LẠI THEO LOGIC CỦA BẠN)
        # Mẫu số là min của K và tổng số kết quả đúng thực tế
        denominator = min(k, total_true_items) 
        p_k = precisions_at_i[:actual_k]
        metrics[f'MAP@{k}'] = np.sum(p_k * r_k) / denominator

    return pd.Series(metrics)

In [46]:
import pandas as pd
import numpy as np
import pickle

# 1. Nạp dữ liệu
with open('groundtruth200.pkl', 'rb') as f:
    loaded_groundtruth = pickle.load(f)

with open('semlex_retrieval_results_title.pkl', 'rb') as f:
    retrieval_results = pickle.load(f)

# (Đảm bảo bạn đã định nghĩa/chạy cell chứa hàm `calculate_metrics_optimized` ở trên)

print("🚀 Bắt đầu quá trình chấm điểm hàng loạt (Ablation Evaluation)...")

final_evaluation = []

# Duyệt qua từng mốc trọng số (0.0 đến 1.0)
for w_bge, queries_dict in retrieval_results.items():
    w_tfidf = round(1.0 - w_bge, 1)
    
    # Mảng chứa kết quả metric của toàn bộ 200 query cho mốc trọng số này
    weight_metrics_list = []
    
    for q_id, retrieved_ids in queries_dict.items():
        # Lấy Ground Truth của query này
        true_set = loaded_groundtruth.get(q_id, set())
        
        # Tạo DataFrame giả lập định dạng 'group' cho hàm metrics
        df_query = pd.DataFrame({
            'item_id': retrieved_ids,
            'rank': np.arange(len(retrieved_ids)) + 1
        })
        
        # Gán nhãn (1 nếu model tìm đúng item nằm trong ground truth, 0 nếu sai)
        df_query['label'] = df_query['item_id'].apply(lambda x: 1 if x in true_set else 0)
        
        # Định nghĩa tên (name) cho DataFrame để hàm đọc được q_id
        df_query.name = q_id
        
        # Gọi hàm tính toán
        metrics_series = calculate_metrics_optimized(df_query, loaded_groundtruth)
        weight_metrics_list.append(metrics_series)
        
    # Tính điểm trung bình của toàn bộ 200 query cho trọng số này
    df_weight_metrics = pd.DataFrame(weight_metrics_list)
    avg_metrics = df_weight_metrics.mean().to_dict()
    
    # Gắn thêm thông tin trọng số vào để làm báo cáo
    avg_metrics['Weight_BGE_M3'] = w_bge
    avg_metrics['Weight_TFIDF'] = w_tfidf
    
    final_evaluation.append(avg_metrics)

# 2. Xây dựng Bảng báo cáo cuối cùng
df_final_report = pd.DataFrame(final_evaluation)

# Sắp xếp lại các cột cho đẹp mắt (Trọng số lên đầu, sau đó đến các chỉ số)
cols = ['Weight_BGE_M3', 'Weight_TFIDF', 
        'MAP', 'MRR', 'Hit@5', 'P@5', 'NDCG@5', 'MRR@5', 'MAP@5',
        'Hit@10', 'P@10', 'NDCG@10', 'MRR@10', 'MAP@10',
        'Hit@20', 'P@20', 'NDCG@20', 'MRR@20', 'MAP@20']
# Nếu bạn muốn xem cả @5 và @10 thì thêm vào list cols ở trên

df_final_report = df_final_report[[c for c in cols if c in df_final_report.columns]]

# Lưu ra Excel để vẽ biểu đồ cho bài báo
df_final_report.to_excel("semlex_ablation_metrics_report.xlsx", index=False)

print("✅ ĐÁNH GIÁ HOÀN TẤT! Dưới đây là kết quả của mô hình SemLex:")
print("-" * 80)
# In ra bảng dữ liệu với định dạng gọn gàng
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_final_report.round(4).to_string(index=False))

🚀 Bắt đầu quá trình chấm điểm hàng loạt (Ablation Evaluation)...
✅ ĐÁNH GIÁ HOÀN TẤT! Dưới đây là kết quả của mô hình SemLex:
--------------------------------------------------------------------------------
 Weight_BGE_M3  Weight_TFIDF    MAP    MRR  Hit@5    P@5  NDCG@5  MRR@5  MAP@5  Hit@10   P@10  NDCG@10  MRR@10  MAP@10  Hit@20   P@20  NDCG@20  MRR@20  MAP@20
           0.0           1.0 0.0580 0.6139 0.8000 0.4712  0.4710 0.5989 0.3808  0.8976 0.4610   0.4670  0.6112  0.3552  0.9317 0.4205   0.4458  0.6139  0.3190
           0.1           0.9 0.0593 0.6239 0.8049 0.4790  0.4802 0.6093 0.3916  0.8878 0.4649   0.4734  0.6202  0.3633  0.9366 0.4278   0.4537  0.6239  0.3265
           0.2           0.8 0.0608 0.6381 0.8098 0.4810  0.4863 0.6243 0.3990  0.8878 0.4683   0.4793  0.6345  0.3689  0.9366 0.4320   0.4596  0.6381  0.3322
           0.3           0.7 0.0626 0.6568 0.8195 0.4907  0.4993 0.6441 0.4135  0.8829 0.4746   0.4883  0.6524  0.3782  0.9463 0.4380   0.4676  0.6568  0.339

In [47]:
df_final_report.to_excel("semlex_title_metrics_report.xlsx", index=False)

# Evaluate all model with 200 queries

In [49]:
import requests
import pandas as pd
import pickle
from tqdm.notebook import tqdm

API_URL = "http://127.0.0.1:8000/api/search"
TOP_K = 20

print("🚀 Đang nạp 200 queries...")
df_queries = pd.read_json("queries_natural_200.json")

# Cấu hình Models (Giữ nguyên từ code cũ)
MODEL_MAPPING = {
    "TF-IDF": "tfidf", "Word2Vec": "w2v", "Doc2Vec": "doc2vec",
    "LaBSE": "labse", "BGE-M3": "bge", "Paraphrase": "mpnet",
    "Ensemble": "ensemble", "Word2Vec_sg": "w2v_sg", "Doc2Vec_dbow": "doc2vec_dbow"
}
TYPE_TO_SEARCH_MODE = {"basic": "title", "upgrade": "overall"}

MODELS_CONFIG = [
    {"name": "TF-IDF", "type": "basic"}, {"name": "TF-IDF", "type": "upgrade"},
    {"name": "Word2Vec", "type": "basic"}, {"name": "Word2Vec", "type": "upgrade"},
    {"name": "Word2Vec_sg", "type": "basic"}, {"name": "Word2Vec_sg", "type": "upgrade"},
    {"name": "Doc2Vec", "type": "basic"}, {"name": "Doc2Vec", "type": "upgrade"},
    {"name": "Doc2Vec_dbow", "type": "basic"}, {"name": "Doc2Vec_dbow", "type": "upgrade"},
    {"name": "LaBSE", "type": "basic"}, {"name": "LaBSE", "type": "upgrade"},
    {"name": "BGE-M3", "type": "basic"}, {"name": "BGE-M3", "type": "upgrade"},
    {"name": "Paraphrase", "type": "basic"}, {"name": "Paraphrase", "type": "upgrade"},
    {"name": "Ensemble", "type": "basic"}, {"name": "Ensemble", "type": "upgrade"},
]

results_log = []

for config in MODELS_CONFIG:
    model_short = MODEL_MAPPING.get(config['name'], config['name'].lower())
    search_mode = TYPE_TO_SEARCH_MODE.get(config['type'], "overall")
    model_full_name = f"{model_short}_{config['type']}"
    
    # Quét qua 200 câu query
    for q_idx, row in tqdm(df_queries.iterrows(), total=len(df_queries), desc=model_full_name, leave=False):
        payload = {
            "query": row['query'],
            "model_name": model_short,  
            "search_type": search_mode,    
            "filters": {}
        }
        
        try:
            response = requests.post(API_URL, json=payload)
            if response.status_code == 200:
                predictions = response.json()
                for rank, item in enumerate(predictions):
                    if rank >= TOP_K: break
                    results_log.append({
                        "model": model_full_name,     # Tên model (VD: bge_upgrade)
                        "query_id": q_idx,
                        "query_text": row['query'],
                        "rank": rank + 1,
                        "item_id": item.get('id'),
                        "score": item.get('similarity_score', item.get('score', 0))
                    })
        except Exception as e:
            pass

# Lưu toàn bộ kết quả thô xuống đĩa
df_all_results = pd.DataFrame(results_log)
df_all_results.to_pickle("all_models_retrieval_200.pkl")
print(f"🎉 Hoàn tất! Đã thu thập {len(df_all_results)} kết quả trả về từ Backend.")

🚀 Đang nạp 200 queries...


tfidf_basic:   0%|          | 0/205 [00:00<?, ?it/s]

tfidf_upgrade:   0%|          | 0/205 [00:00<?, ?it/s]

w2v_basic:   0%|          | 0/205 [00:00<?, ?it/s]

w2v_upgrade:   0%|          | 0/205 [00:00<?, ?it/s]

...

mpnet_basic:   0%|          | 0/205 [00:00<?, ?it/s]

mpnet_upgrade:   0%|          | 0/205 [00:00<?, ?it/s]

ensemble_basic:   0%|          | 0/205 [00:00<?, ?it/s]

ensemble_upgrade:   0%|          | 0/205 [00:00<?, ?it/s]

🎉 Hoàn tất! Đã thu thập 57400 kết quả trả về từ Backend.


In [ ]:
import pandas as pd
import numpy as np
import pickle

# =====================================================================
# 1. HÀM TÍNH METRICS CỦA BẠN (GIỮ NGUYÊN 100%)
# =====================================================================
def calculate_metrics_optimized(group, ground_truth):
    if isinstance(group.name, tuple):
        model_name, q_id = group.name
    else:
        q_id = group.name
    
    group = group.sort_values('rank')
    relevants = group['label'].values 
    
    true_set = ground_truth.get(q_id, set())
    total_true_items = len(true_set)
    
    metrics = {}
    
    if total_true_items == 0:
        return pd.Series({ 'MAP': 0, 'MRR': 0, 'Hit@5': 0, 'P@5': 0, 'NDCG@5': 0, 'MRR@5': 0, 'MAP@5': 0,
                           'Hit@10': 0, 'P@10': 0, 'NDCG@10': 0, 'MRR@10': 0, 'MAP@10': 0,
                           'Hit@20': 0, 'P@20': 0, 'NDCG@20': 0, 'MRR@20': 0, 'MAP@20': 0 })

    ranks = np.arange(len(relevants)) + 1
    cumulative_relevant = np.cumsum(relevants)
    precisions_at_i = cumulative_relevant / ranks
    discounts = np.log2(ranks + 1)
    
    metrics['MAP'] = np.sum(precisions_at_i * relevants) / total_true_items

    try:
        first_rel_idx = np.argmax(relevants == 1)
        if relevants[first_rel_idx] == 1:
            metrics['MRR'] = 1 / (first_rel_idx + 1)
        else:
            metrics['MRR'] = 0
    except:
        metrics['MRR'] = 0

    cutoffs = [5, 10, 20]
    for k in cutoffs:
        actual_k = min(k, len(relevants))
        r_k = relevants[:actual_k]
        
        metrics[f'Hit@{k}'] = 1 if np.any(r_k) else 0
        metrics[f'P@{k}'] = cumulative_relevant[actual_k - 1] / k if actual_k > 0 else 0
        
        dcg = np.sum(r_k / discounts[:actual_k])
        ideal_r_k = np.zeros(actual_k)
        num_ones_possible = min(actual_k, total_true_items)
        ideal_r_k[:num_ones_possible] = 1 
        idcg = np.sum(ideal_r_k / discounts[:actual_k])
        metrics[f'NDCG@{k}'] = dcg / idcg if idcg > 0 else 0
            
        first_idx_k = np.argmax(r_k == 1)
        if r_k[first_idx_k] == 1:
            metrics[f'MRR@{k}'] = 1 / (first_idx_k + 1)
        else:
            metrics[f'MRR@{k}'] = 0
            
        denominator = min(k, total_true_items) 
        p_k = precisions_at_i[:actual_k]
        metrics[f'MAP@{k}'] = np.sum(p_k * r_k) / denominator

    return pd.Series(metrics)

# =====================================================================
# 2. XỬ LÝ DỮ LIỆU & ĐÁNH GIÁ (EVALUATION PIPELINE)
# =====================================================================
print("🚀 Đang nạp dữ liệu Ground Truth và Kết quả truy xuất...")

# Nạp Ground Truth
with open('groundtruth200.pkl', 'rb') as f:
    ground_truth = pickle.load(f)

# Nạp file kết quả đã chạy (giả sử tên file là all_models_retrieval_200.pkl)
# Nếu bạn lưu tên khác thì sửa lại ở đây
df_results = pd.read_pickle("all_models_retrieval_200.pkl")

# --- A. LOẠI BỎ DOC2VEC ---
print("🧹 Đang loại bỏ các model Doc2Vec bị lỗi...")
df_results = df_results[~df_results['model'].str.contains('doc2vec', case=False, na=False)]


# --- B. GÁN NHÃN ĐÚNG/SAI ---
print("⚙️ Đang gán nhãn (Labeling) dựa trên Ground Truth...")
def assign_label(row):
    true_set = ground_truth.get(row['query_id'], set())
    return 1 if row['item_id'] in true_set else 0

df_results['label'] = df_results.apply(assign_label, axis=1)

# --- C. TÍNH ĐIỂM CHO TỪNG QUERY CỦA TỪNG MODEL ---
print("📊 Đang tính toán Metrics...")
raw_metrics = df_results.groupby(['model', 'query_id']).apply(
    lambda g: calculate_metrics_optimized(g, ground_truth)
)

# --- D. XỬ LÝ LỖI "TRỐNG KẾT QUẢ" (RẤT QUAN TRỌNG) ---
# Đảm bảo nếu model nào không tìm ra job nào cho 1 query, nó vẫn nhận điểm 0 cho query đó
valid_models = df_results['model'].unique()
all_query_ids = list(ground_truth.keys())

# Tạo lưới (Grid) toàn bộ tổ hợp Model x Query
multi_idx = pd.MultiIndex.from_product([valid_models, all_query_ids], names=['model', 'query_id'])
raw_metrics = raw_metrics.reindex(multi_idx).fillna(0).reset_index()

# --- E. TÍNH TRUNG BÌNH & TÁCH CỘT MODE (TITLE / OVERALL) ---
final_summary = raw_metrics.groupby('model').mean().reset_index()

# Hàm tách tên Model và Mode
def parse_model_name(full_name):
    if '_basic' in full_name:
        return full_name.replace('_basic', ''), 'title'
    elif '_upgrade' in full_name:
        return full_name.replace('_upgrade', ''), 'overall'
    return full_name, 'unknown'

final_summary['Algorithm'] = final_summary['model'].apply(lambda x: parse_model_name(x)[0].upper())
final_summary['Search_Mode'] = final_summary['model'].apply(lambda x: parse_model_name(x)[1].upper())

# --- F. TRÌNH BÀY BÁO CÁO ---
# Bỏ đi các cột dư thừa
cols_to_drop = ['query_id', 'model']
final_summary = final_summary.drop(columns=[c for c in cols_to_drop if c in final_summary.columns])

# Sắp xếp lại thứ tự cột cho đẹp (Tên Model -> Mode -> Các chỉ số)
first_cols = ['Algorithm', 'Search_Mode']
metric_cols = [c for c in final_summary.columns if c not in first_cols]
final_summary = final_summary[first_cols + metric_cols]

# Sắp xếp danh sách ưu tiên xem Mode overall trước, và theo MAP@20 giảm dần
final_summary = final_summary.sort_values(by=['Search_Mode', 'MAP@20'], ascending=[False, False])

# Lưu ra Excel
final_summary.to_excel("final_models_evaluation.xlsx", index=False)

print("✅ ĐÁNH GIÁ HOÀN TẤT! File báo cáo 'semlex_final_models_evaluation.xlsx' đã sẵn sàng.")
print("-" * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(final_summary.round(4))

🚀 Đang nạp dữ liệu Ground Truth và Kết quả truy xuất...
🧹 Đang loại bỏ các model Doc2Vec bị lỗi...
⚙️ Đang gán nhãn (Labeling) dựa trên Ground Truth...
📊 Đang tính toán Metrics...
✅ ĐÁNH GIÁ HOÀN TẤT! File báo cáo 'semlex_final_models_evaluation.xlsx' đã sẵn sàng.
----------------------------------------------------------------------------------------------------


,Algorithm,Search_Mode,MAP,MRR,Hit@5,P@5,NDCG@5,MRR@5,MAP@5,Hit@10,P@10,NDCG@10,MRR@10,MAP@10,Hit@20,P@20,NDCG@20,MRR@20,MAP@20
2,ENSEMBLE,TITLE,0.0624,0.7019,0.8585,0.5473,0.5603,0.6929,0.4827,0.9122,0.4976,0.5242,0.7002,0.4212,0.9366,0.4437,0.4861,0.7019,0.3639
8,TFIDF,TITLE,0.0580,0.6139,0.8000,0.4712,0.4710,0.5989,0.3808,0.8976,0.4610,0.4670,0.6112,0.3552,0.9317,0.4205,0.4458,0.6139,0.3190
0,BGE,TITLE,0.0422,0.6522,0.8000,0.4605,0.4806,0.6430,0.4008,0.8390,0.4112,0.4407,0.6485,0.3372,0.8927,0.3661,0.4047,0.6522,0.2856
4,LABSE,TITLE,0.0246,0.5372,0.6634,0.3473,0.3687,0.5213,0.2955,0.7512,0.3068,0.3349,0.5333,0.2404,0.8098,0.2659,0.3000,0.5372,0.1940
11,W2V_SG,TITLE,0.0143,0.4112,0.5463,0.2868,0.2878,0.3890,0.2264,0.6732,0.2610,0.2700,0.4063,0.1898,0.7463,0.2315,0.2481,0.4112,0.1589
6,MPNET,TITLE,0.0077,0.2803,0.3610,0.1649,0.1716,0.2562,0.1304,0.4927,0.1615,0.1672,0.2736,0.1096,0.5902,0.1444,0.1540,0.2803,0.0889
10,W2V,TITLE,0.0004,0.0689,0.0683,0.0302,0.0296,0.0424,0.0220,0.2927,0.0600,0.0492,0.0663,0.0263,0.3317,0.0451,0.0426,0.0689,0.0199
3,ENSEMBLE,OVERALL,0.0701,0.7183,0.8390,0.5356,0.5547,0.7039,0.4745,0.9268,0.5132,0.5360,0.7162,0.4259,0.9561,0.4727,0.5124,0.7183,0.3812
1,BGE,OVERALL,0.0454,0.6549,0.7707,0.4507,0.4727,0.6389,0.3918,0.8488,0.4385,0.4590,0.6500,0.3511,0.9171,0.4046,0.4361,0.6549,0.3085
9,TFIDF,OVERALL,0.0555,0.6308,0.8244,0.4546,0.4618,0.6159,0.3640,0.9073,0.4171,0.4361,0.6277,0.3128,0.9463,0.3978,0.4279,0.6308,0.2831
